# L13 · 闭环评估与 Capstone

本实验沿讲义中的六道证据门线性推进：运行环境、数据集、checkpoint、冻结协议、实时 rollout，以及结论范围明确的报告。每一步只生产下一步所需的证据。

核心路径先对一个 checkpoint 运行 5 个参考条件回合。可选的 Capstone 再使用相同 seed 运行一个预先声明的 held-out 条件。5 个回合只是集成 smoke，不是稳定的性能基准。

## 如何使用这份 notebook

按照编号步骤从上到下依次完成。

1. 只在开始时编辑一次配置单元。把 `checkpoint` 设为自己的本地路径或固定 revision 的 Hugging Face mapping；保持为 `None` 时使用同一单元中明确声明的课程默认 checkpoint。让 `dataset_root` 指向配套的本地数据集；如果目录不存在，第 1 步会使用配置单元中可见的固定 dataset fallback。每次重新进行正式运行时，都要选择一个新的 `run_name`。
2. 运行第 1–3 步，在查看结果前审计 artifact 并冻结比较设计。第 1 步可能下载 dataset metadata，但不会下载 episodes 或视频，也不会启动 Genesis。
3. 当第 1 步打印 `Ready to run Step 4: True` 后继续。运行第 4 步会下载或加载 checkpoint，并立即执行参考条件回合。
4. 在第 5 步审计视频和逐回合记录。
5. 要继续 Capstone，就运行第 6 步；然后先运行一次第 7 步，查看 evidence card 和句式提示。在同一个 cell 中填写三个字段，再只重跑第 7 步来写出报告。

所有实验选择都来自可见的配置。运行评估 cell 本身就是启动对应子进程的明确动作，因此 Genesis 不会在 notebook kernel 中初始化。

In [ ]:
# Edit artifact paths and the declared experiment here.
CONFIG = {
    # A local pretrained_model directory, a pinned Hugging Face mapping, or None.
    # None uses COURSE_DEFAULT_CHECKPOINT below.
    # Example: {'repo_id': 'org/repo', 'revision': '<40-hex-commit>',
    #           'subfolder': 'checkpoints/020000/pretrained_model'}
    'checkpoint': None,
    # Prefer this local dataset when it exists. Otherwise Step 1 downloads
    # COURSE_DEFAULT_DATASET metadata from Hugging Face.
    'dataset_repo_id': 'genesis/fruit_pick',
    'dataset_root': 'datasets/fruit_pick',
    'run_name': 'fruit-pick-hf-fallback',
    'gpu_index': 0,
    'policy_device': 'cuda',
    'render': True,
    'protocol': {
        'episodes': 5,
        'base_seed': 1000,
        'control_fps': 100,
        'dwell_seconds': 0.4,
        'max_seconds': 15.0,
        'pick_objects': ('011_banana',),
        'place_target': '024_bowl',
        'success_tolerance_m': 0.06,
    },
    'held_out': {
        'name': 'shifted-friction-0_9',
        'factor': 'friction_ratio',
        'evaluation_range': (0.9, 0.9),
        'training_support': None,
    },
}

# Used only when CONFIG['dataset_root'] does not exist locally.
# L13 downloads meta/** only; policy evaluation does not read episodes or videos.
COURSE_DEFAULT_DATASET = {
    'repo_id': 'wangxun1010/fruit_pick',
    'revision': '475e8c6ec7079c4c52b20a0221bf959acb541abd',
    'subfolder': '.',
}

# Used only when CONFIG['checkpoint'] is None. The commit pin makes the download reproducible.
COURSE_DEFAULT_CHECKPOINT = {
    'repo_id': 'wangxun1010/smolvla-fruit-pick-0821',
    'revision': 'cdf720f162e6ee86dbc856cdf15c5dcd4a673221',
    'subfolder': '.',
}

## 第 1 步——审计运行环境、数据集与 checkpoint 来源

Capstone 的前三道证据门都是身份检查。数据集与 checkpoint 必须在训练数据身份、相机特征、state/action 维度、FPS、processors 和相机重命名等方面一致。第 1 步优先使用 `CONFIG['dataset_root']`；目录不存在时，只从固定的 `COURSE_DEFAULT_DATASET` 下载 `meta/**`。本地 checkpoint 会在这里审计；固定 revision 的远端 checkpoint 会标记为 `DEFERRED`，并在第 4 步下载后完成审计。

这一步只读取 metadata。固定 revision 的 Hugging Face checkpoint 只有在你运行第 4 步时才会下载。Runtime summary 还会打印由 `render`、`gpu_index` 和 `policy_device` 推导出的确切子进程环境。

In [ ]:
import json

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.evaluation_contract import (
    evaluate_predicate_trace,
    validate_evaluation_protocol,
)
from robo_genesis.l13_workflow import (
    audit_l13_evidence,
    audit_l13_inputs,
    build_l13_capstone_guidance,
    capstone_statements,
    compare_l13_runs,
    declare_l13_conditions,
    prepare_l13_workflow,
    require_l13_inputs,
    run_l13_condition,
    runtime_summary,
    write_l13_capstone_report,
)

lesson = load_course_manifest().lesson('L13')
assert lesson.status.value == 'gpu-verified'
workflow = prepare_l13_workflow(
    CONFIG,
    course_default_dataset=COURSE_DEFAULT_DATASET,
    course_default_checkpoint=COURSE_DEFAULT_CHECKPOINT,
)
preflight = audit_l13_inputs(
    workflow, allow_dataset_download=True, allow_checkpoint_download=False
)

print(json.dumps(runtime_summary(workflow, lesson_status=lesson.status.value), indent=2, default=str))
print(json.dumps(list(preflight.rows), indent=2))
print('Ready to run Step 4:', preflight.ready_for_reference)
if not preflight.ready_for_reference:
    print("STOP before Step 4 — edit CONFIG above and fix every FAIL row.")

## 第 2 步——冻结任务与成功协议

任何 rollout 结果出现前都要先冻结协议。L08 containment predicate 必须连续满足 `ceil(dwell_seconds × policy_fps)` 次检查才算成功；超时处最后一帧为真，也不能绕过 dwell 规则。

**协议练习：**修改 `SYNTHETIC_PREDICATE_TRACE`，先在纸上预测最大连续次数与回合结果，再运行单元。至少包含一条最后一帧为真、但连续次数不足的 trace。

In [ ]:
AVAILABLE_OBJECTS = (
    '003_cracker_box', '004_sugar_box', '005_tomato_soup_can',
    '006_mustard_bottle', '007_tuna_fish_can', '008_pudding_box',
    '009_gelatin_box', '010_potted_meat_can', '011_banana',
    '019_pitcher_base', '021_bleach_cleanser', '024_bowl',
)
protocol_config = CONFIG['protocol']
policy_fps = float(preflight.dataset['fps']) if preflight.dataset else 5.0
reference_protocol = validate_evaluation_protocol(
    episodes=protocol_config['episodes'],
    policy_fps=policy_fps,
    control_fps=protocol_config['control_fps'],
    dwell_seconds=protocol_config['dwell_seconds'],
    max_seconds=protocol_config['max_seconds'],
    tolerance=protocol_config['success_tolerance_m'],
    pick_objects=protocol_config['pick_objects'],
    place_target=protocol_config['place_target'],
    available_objects=AVAILABLE_OBJECTS,
    base_seed=protocol_config['base_seed'],
    balanced=True,
    condition_name='reference',
    render_enabled=workflow.render_enabled,
)

SYNTHETIC_PREDICATE_TRACE = [False, True, False, True]
trace_evidence = evaluate_predicate_trace(
    SYNTHETIC_PREDICATE_TRACE,
    dwell_seconds=protocol_config['dwell_seconds'],
    policy_fps=policy_fps,
)
print(json.dumps({
    'task': reference_protocol['task'],
    'timing': reference_protocol['timing'],
    'success_rule': reference_protocol['success_rule'],
    'episode_seeds': reference_protocol['sampling']['episode_seeds'],
    'object_schedule': reference_protocol['sampling']['object_schedule'],
    'synthetic_trace': trace_evidence,
}, indent=2))

## 第 3 步——预先声明参考条件与一个 held-out 条件

Capstone 比较只改变一个因素，同时固定任务、相机合同、对象调度和 episode seed。条件标签必须相对于训练 provenance：缺失支持范围或评估范围与其重叠时保持 `shifted`；只有文档证明二者不相交时才能称为 OOD。

**单因素练习：**只修改 `CONFIG['held_out']`。运行前先解释得到的标签，并列出所有保持不变的因素。

In [ ]:
reference_condition, held_out_condition = declare_l13_conditions(CONFIG['held_out'])
condition_plan = {
    'reference': reference_condition,
    'held_out': held_out_condition or 'SKIP — no changed factor was declared',
}
print(json.dumps(condition_plan, indent=2))

## 第 4 步——运行 5 回合参考条件评估

这是第一个高成本步骤。确认第 1–3 步正确后再运行这个 cell。需要时它会下载固定 revision 的 checkpoint，在独立进程中启动 `robo_genesis.eval_policy`，捕获完整日志，映射选定 GPU，并要求使用新的结果与视频位置。Evaluator 崩溃会中止单元，不会被记成普通策略失败。

In [ ]:
reference_inputs = require_l13_inputs(workflow)
reference_run = run_l13_condition(
    workflow, reference_inputs, reference_protocol, reference_condition
)
print(json.dumps({
    'status': 'COMPLETED',
    'results': str(reference_run.paths['results']),
    'videos': str(reference_run.paths['videos']),
    'summary': reference_run.report['summary'],
}, indent=2))

## 第 5 步——审计逐回合记录与 rollout 视频

只有当 `k/n`、Wilson 区间、终止原因计数、分对象计数、seed、协议和条件都能从 episode rows 重新计算时，才接受保存的 summary。每个回合还必须提供可解码的 H.264 视频，并通过尺寸和帧数检查。

**失败证据练习：**完成真实运行后，使用观看前确定的选择规则标注一个失败 seed。记录简短类别和 rollout 中直接可见的证据；不要根据汇总成功率推断训练数据或模型架构原因。

In [ ]:
MANUAL_FAILURE_LABELS = {
    # 1003: {
    #     'category': 'release timing',
    #     'visible_evidence': 'the object leaves the gripper beside the bowl',
    # },
}
reference_evidence = audit_l13_evidence(
    reference_run.report, manual_failure_labels=MANUAL_FAILURE_LABELS
)
print(json.dumps({
    'summary': reference_evidence['summary'],
    'videos': reference_evidence['videos'],
    'selection_rule': reference_evidence['selection_rule'],
    'manual_failure_labels': reference_evidence['manual_failure_labels'],
}, indent=2))
from IPython.display import Video, display
display(Video(reference_evidence['representative_episode']['video'], embed=True))

## 第 6 步——运行 matched held-out 比较

只有参考条件证据完整、且你希望继续 Capstone 时才运行这个 cell。它复用 checkpoint、任务、对象调度和 5 个 seed，只改变第 3 步声明的一个因素。报告同时给出两组汇总 `k/n` 与每个 seed 的结果翻转。5 对 matched 样本可以指导诊断，但不能证明统计显著性。

In [ ]:
if held_out_condition is None:
    raise RuntimeError('Declare one held-out factor in Step 3 before Step 6')
held_out_run = run_l13_condition(
    workflow, reference_inputs, reference_protocol, held_out_condition
)
held_out_evidence = audit_l13_evidence(held_out_run.report)
paired_changes = compare_l13_runs(reference_run, held_out_run)
print(json.dumps({
    'reference': reference_evidence['summary'],
    held_out_condition['name']: held_out_evidence['summary'],
    'paired_outcome_changes': paired_changes,
    'interpretation': 'Matched outcomes; no significance claim is made.',
}, indent=2))

## 第 7 步——沿证据链撰写 Capstone 结论

报告回答一个范围明确的问题，并追溯到运行环境、数据集、checkpoint、冻结协议、逐回合记录、视频、限制和复现命令。完成第 6 步后先运行一次这个 cell：它会根据真实结果打印 evidence card 和句式提示。然后在同一个 cell 的 `CAPSTONE_CONCLUSION` 中填写三个空字段，再只重跑第 7 步；rollout 不会重复执行。

**Capstone 练习：**根据 evidence card 说明当前运行支持什么、不能支持什么，以及哪一项后续实验最能减少剩余的不确定性。不要编造 episode rows 和视频中没有出现的因果解释。

In [ ]:
reference_summary = reference_evidence['summary']
held_out_summary = held_out_evidence['summary']
guidance = build_l13_capstone_guidance(
    reference_summary, held_out_summary, held_out_condition, paired_changes
)
print('EVIDENCE CARD — base every conclusion on these observed values')
print(json.dumps(guidance['evidence_card'], indent=2))
print('\nSENTENCE STEMS — edit these into your own scoped conclusions')
print(json.dumps(guidance['sentence_stems'], indent=2))

# First run: leave these blank and read the evidence card above.
# Then replace all three blanks with your conclusions and rerun only this cell.
CAPSTONE_CONCLUSION = {
    'question': guidance['question'],
    'supported_claim': '',
    'unsupported_claim': '',
    'next_evidence': '',
}
required_fields = ('supported_claim', 'unsupported_claim', 'next_evidence')
missing_fields = [
    name for name in required_fields if not CAPSTONE_CONCLUSION[name].strip()
]
statements = capstone_statements(CAPSTONE_CONCLUSION)
capstone_output = None
if missing_fields:
    print("\nACTION REQUIRED — Step 7 is a two-pass cell.")
    print('Fill these CAPSTONE_CONCLUSION fields:', ', '.join(missing_fields))
    print('Use the exact k/n and matched changes above; do not invent causes.')
    print('After editing this cell, rerun only Step 7. Rollouts are not repeated.')
else:
    capstone_output = write_l13_capstone_report(
        workflow, reference_run, held_out_run, paired_changes, statements
    )
    print(capstone_output['markdown'])

core_complete = bool(
    reference_summary['episodes'] == protocol_config['episodes']
    and reference_run.report['protocol']['render_enabled']
    and reference_run.report['environment'].get('actual_policy_device', '').startswith('cuda')
    and len(reference_evidence['videos']) == protocol_config['episodes']
)
capstone_complete = bool(
    core_complete
    and len(held_out_evidence['videos']) == protocol_config['episodes']
    and len(paired_changes) == protocol_config['episodes']
    and capstone_output is not None
)
print('Step 1 preflight: INPUTS READY')
print('Core Reference evaluation:', 'COMPLETED' if core_complete else 'NOT COMPLETED')
print('Capstone comparison and report:', 'COMPLETED' if capstone_complete else 'NOT COMPLETED')

## 核心实验之外——checkpoint sweep

讲义把 `python -m robo_genesis.eval_sweep` 作为可选的终端扩展。这里有意不再增加 notebook 分支：它的成本是 `checkpoint 数量 × episodes`，单 checkpoint Capstone 不要求它，而且一个 checkpoint 只能形成一个点，不能称为学习曲线。